# Is the RPI Fair? Code and live demo
**Every line of code we use is in this notebook.** Cell 1 copies the public repository (https://github.com/JiaheLi0604/apma4903-rpi-trueskill), which only supplies data: the 2022–2025 match table (from ncaavolleyballr) and the saved season predictions.

**Final model (Nov 20):** a team's expected lineup counts only players who have played this season (before the team's first match of a season: last season's players), and a new player starts at her team's current average rating instead of 25. The previous version is reproducible with `lineup='all', new_prior='global'`.

Run the cells in order. Sections 1–5 take a few seconds each. Section 6 (rebuilding the season predictions) takes several minutes and is optional.

In [ ]:
# Copy the public repository (data + saved predictions). Every line of CODE is in this notebook.
import os
if not os.path.exists('apma4903-rpi-trueskill'):
    !git clone -q https://github.com/JiaheLi0604/apma4903-rpi-trueskill.git
%cd apma4903-rpi-trueskill

## 1. The algorithm: Algorithm 1 (MessagePassing, one match)
Gaussians are stored in precision form (pi = 1/var, tau = mu/var), so multiplying adds and dividing subtracts. Each `# Algorithm 1, line k` comment matches line k of the Algorithm 1 slide.


In [ ]:
import numpy as np
from scipy.special import log_ndtr

LOG_SQRT_2PI = 0.9189385332046727

class G:
    """A Gaussian in precision form."""
    def __init__(s, pi=0.0, tau=0.0): s.pi, s.tau = float(pi), float(tau)
    @staticmethod
    def mv(mu, var): return G(1.0/var, mu/var)
    @property
    def mu(s): return s.tau/s.pi
    @property
    def var(s): return 1.0/s.pi
    def __mul__(s, o): return G(s.pi+o.pi, s.tau+o.tau)        # product of Gaussians: precisions add
    def __truediv__(s, o): return G(s.pi-o.pi, s.tau-o.tau)    # division: precisions subtract
    def __repr__(s): return f"N(mean={s.mu:.4f}, sd={np.sqrt(s.var):.4f})"

def VW(t):
    v = float(np.exp(-0.5*t*t - LOG_SQRT_2PI - log_ndtr(t))); return v, v*(v+t)

def message_passing(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, log=None):
    """Return posterior (mu, var) for every player of A and B.  `log` (a list) collects every message."""
    def L(name, g):
        if log is not None: log.append((name, g.mu, g.var))
        return g
    b2 = beta*beta
    if not A_won:                                         # relabel so that the winner is always 'A'
        (mB2, vB2), (mA2, vA2) = message_passing(muB, varB, wB, muA, varA, wA, beta, -h, True, log)
        return (mA2, vA2), (mB2, vB2)
    teams = {"A": (np.asarray(muA,float), np.asarray(varA,float), np.asarray(wA,float)),
             "B": (np.asarray(muB,float), np.asarray(varB,float), np.asarray(wB,float))}
    # Algorithm 1, line 1: skill -> performance (add beta^2)   [the new-season drift is applied by the season engine, Algorithm 2]
    prior, perf_down, team_down = {}, {}, {}
    for k,(m,v,w) in teams.items():
        prior[k] = [L(f"prior->s_{k}{i+1}", G.mv(m[i], v[i])) for i in range(len(m))]
        perf_down[k] = [L(f"s_{k}{i+1}->p_{k}{i+1}", G.mv(g.mu, g.var+b2)) for i,g in enumerate(prior[k])]
        # Algorithm 1, line 1: team = weighted sum: mean = sum w*mu, var = sum w^2*var  (h enters the gap, line 2)
        tm = sum(w[i]*perf_down[k][i].mu for i in range(len(m))) + (h if k=="A" else 0.0)
        tv = sum(w[i]**2*perf_down[k][i].var for i in range(len(m)))
        team_down[k] = L(f"sum->t_{k}", G.mv(tm, tv))
    # Algorithm 1, line 2: d = A - B (+h): means subtract, variances add
    d_in = L("diff->d", G.mv(team_down["A"].mu-team_down["B"].mu, team_down["A"].var+team_down["B"].var))
    # Algorithm 1, line 3: A won, so cut d < 0 and moment-match with V, W (Nugget 2)
    c = np.sqrt(d_in.var); t = d_in.mu/c; V, W = VW(t)
    d_post = L("d marginal (moment-matched)", G.mv(d_in.mu + c*V, d_in.var*(1-W)))
    # Algorithm 1, line 4: divide out the prediction, keep only the news
    d_up = L("trunc->d (= marginal / incoming)", d_post / d_in)
    # Algorithm 1, lines 5-6: back up: A = d + B, B = A - d
    team_up = {"A": L("diff->t_A", G.mv(d_up.mu + team_down["B"].mu, d_up.var + team_down["B"].var)),
               "B": L("diff->t_B", G.mv(team_down["A"].mu - d_up.mu, d_up.var + team_down["A"].var))}
    out = {}
    for k,(m,v,w) in teams.items():
        tu = team_up[k]; base = tu.mu - (h if k=="A" else 0.0)
        post_mu, post_var = [], []
        for i in range(len(m)):
            others_m = sum(w[j]*perf_down[k][j].mu for j in range(len(m)) if j!=i)
            others_v = sum(w[j]**2*perf_down[k][j].var for j in range(len(m)) if j!=i)
            # Algorithm 1, lines 7-8: for each player, solve the team sum: (team - teammates) / w_i
            p_up = L(f"sum->p_{k}{i+1}", G.mv((base-others_m)/w[i], (tu.var+others_v)/w[i]**2))
            # Algorithm 1, line 8: performance -> skill (add beta^2);  line 9: posterior = prior x news (precisions add)
            s_up = L(f"p_{k}{i+1}->s_{k}{i+1}", G.mv(p_up.mu, p_up.var+b2))
            post = L(f"posterior s_{k}{i+1}", prior[k][i]*s_up)
            post_mu.append(post.mu); post_var.append(post.var)
        out[k] = (np.array(post_mu), np.array(post_var))
    return out["A"], out["B"]

## 2. The two independent checks
`closed_form`: Nugget 2's update formula, with player weights.
`brute_force_grid`: Bayes' rule computed directly on a grid (prior × P(A wins | skills)); no Gaussian approximation.
`monte_carlo`: the same Bayes' rule with random samples, for 6 vs. 6 where a grid is too large.

In [ ]:
from scipy.special import ndtr

def closed_form(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True):
    """Nugget 2's formulas, written for weighted teams."""
    muA,varA,wA,muB,varB,wB = map(lambda x: np.asarray(x,float), (muA,varA,wA,muB,varB,wB))
    b2 = beta*beta
    mu_d = wA@muA - wB@muB + h; c2 = (wA**2)@(varA+b2) + (wB**2)@(varB+b2)
    s = 1.0 if A_won else -1.0; c = np.sqrt(c2); V, W = VW(s*mu_d/c)
    nA = (muA + s*wA*varA/c*V, varA*(1-wA**2*varA/c2*W))
    nB = (muB - s*wB*varB/c*V, varB*(1-wB**2*varB/c2*W))
    return nA, nB

def brute_force_grid(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, n=None, half_width=7.0):
    """Exact posterior mean/var of every skill by direct numerical integration on a grid.
    The performance noise is integrated analytically: P(A wins | skills) = Phi((sum wA sA - sum wB sB + h)/sqrt(beta^2 sum w^2)).
    Everything else is a plain sum over grid points: no Gaussian approximation anywhere."""
    from scipy.special import ndtr
    mus = np.r_[muA, muB]; sds = np.sqrt(np.r_[varA, varB]); ws = np.r_[wA, -np.asarray(wB,float)]
    k = len(mus); n = n or {2: 801, 4: 61}.get(k, 41)
    axes = [np.linspace(m-half_width*s, m+half_width*s, n) for m,s in zip(mus, sds)]
    grids = np.meshgrid(*axes, indexing="ij", sparse=True)
    logprior = sum(-0.5*((g-m)/s)**2 for g,m,s in zip(grids, mus, sds))
    diff = sum(w*g for w,g in zip(ws, grids)) + h
    lik = ndtr((1 if A_won else -1)*diff/np.sqrt(beta**2*np.sum(ws**2)))
    wgt = np.exp(logprior - logprior.max())*lik; Z = wgt.sum()
    means, variances = [], []
    for g in grids:
        m = (wgt*g).sum()/Z; means.append(m); variances.append((wgt*(g-m)**2).sum()/Z)
    k1 = len(muA)
    return (np.array(means[:k1]), np.array(variances[:k1])), (np.array(means[k1:]), np.array(variances[k1:]))

def monte_carlo(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, N=4_000_000, seed=0):
    """Same exact posterior, by sampling skills from the prior and weighting by P(result | skills)."""
    from scipy.special import ndtr
    rng = np.random.default_rng(seed)
    mus = np.r_[muA, muB]; sds = np.sqrt(np.r_[varA, varB]); ws = np.r_[wA, -np.asarray(wB,float)]
    S = mus + sds*rng.standard_normal((N, len(mus)))
    lik = ndtr((1 if A_won else -1)*(S@ws + h)/np.sqrt(beta**2*np.sum(ws**2)))
    Z = lik.sum(); m = (lik@S)/Z; v = (lik@(S-m)**2)/Z
    k1 = len(muA)
    return (m[:k1], v[:k1]), (m[k1:], v[k1:])

## 3. The 2 vs. 2 toy match
Change any number and rerun this cell.

In [ ]:
beta = 25/6        # match-day noise
h = 0.0            # home advantage for team A (0 = neutral site)
A_won = True

#         means      variances (sigma^2)   weights
muA, varA, wA = [28, 22], [3**2, 6**2], [1, 1]
muB, varB, wB = [26, 24], [4**2, 2**2], [1, 1]

log = []
(mA, vA), (mB, vB) = message_passing(muA, varA, wA, muB, varB, wB, beta, h, A_won, log)
print("Every message (mean, sd):")
for name, m, v in log:
    print(f"  {name:35s} {m:8.2f} {np.sqrt(v):6.2f}")

print("\nPlayer   before          after")
rows = [("A1", muA[0], varA[0], mA[0], vA[0]), ("A2", muA[1], varA[1], mA[1], vA[1]),
        ("B1", muB[0], varB[0], mB[0], vB[0]), ("B2", muB[1], varB[1], mB[1], vB[1])]
for lab, m0, v0, m1, v1 in rows:
    print(f"  {lab}   {m0:5.2f} ± {np.sqrt(v0):4.2f}   {m1:5.2f} ± {np.sqrt(v1):4.2f}")

cf = closed_form(muA, varA, wA, muB, varB, wB, beta, h, A_won)
bf = brute_force_grid(muA, varA, wA, muB, varB, wB, beta, h, A_won)
diff = lambda a, b: max(np.abs(np.r_[a[0][0]-b[0][0], a[1][0]-b[1][0]]).max(),
                        np.abs(np.sqrt(np.r_[a[0][1], a[1][1]]) - np.sqrt(np.r_[b[0][1], b[1][1]])).max())
mp_out = ((mA, vA), (mB, vB))
print(f"\nmax difference vs. Nugget 2 formula: {diff(mp_out, cf):.1e}")
print(f"max difference vs. grid integration: {diff(mp_out, bf):.1e}")

## 4. Verification table: three methods on four test matches

In [ ]:
import numpy as np
beta = 25/6
cases = {
 "1v1 two newcomers":      ([25],[8.333**2],[1],      [25],[8.333**2],[1], 0, True),
 "1v1 veteran vs newcomer":([25],[2.0**2],[1],        [25],[8.333**2],[1], 0, True),
 "1v1 upset (weak wins)":  ([20],[3.0**2],[1],        [30],[3.0**2],[1],  0, True),
 "2v2 weighted, home":     ([26,22],[3**2,6**2],[1.2,0.8], [27,24],[4**2,2**2],[1.0,1.0], 2.0, False),
}
for name,(mA,vA,wA,mB,vB,wB,h,won) in cases.items():
    mp_ = message_passing(mA,vA,wA,mB,vB,wB,beta,h,won)
    cf  = closed_form(mA,vA,wA,mB,vB,wB,beta,h,won)
    bf  = brute_force_grid(mA,vA,wA,mB,vB,wB,beta,h,won)
    print(f"\n== {name}")
    for side,k in (("A",0),("B",1)):
        for i in range(len(mp_[k][0])):
            print(f"  {side}{i+1}: MP mu={mp_[k][0][i]:.4f} sd={np.sqrt(mp_[k][1][i]):.4f} | CF mu={cf[k][0][i]:.4f} sd={np.sqrt(cf[k][1][i]):.4f} | GRID mu={bf[k][0][i]:.4f} sd={np.sqrt(bf[k][1][i]):.4f}")
    mpcf = max(np.max(np.abs(np.r_[mp_[0][0]-cf[0][0], mp_[1][0]-cf[1][0]])), np.max(np.abs(np.sqrt(np.r_[mp_[0][1],mp_[1][1]])-np.sqrt(np.r_[cf[0][1],cf[1][1]]))))   # mean or sd, same metric as the grid check
    print(f"  max |MP-CF| = {mpcf:.2e}")
print()
for name,(mA,vA,wA,mB,vB,wB,h,won) in cases.items():
    mp_=message_passing(mA,vA,wA,mB,vB,wB,beta,h,won); bf=brute_force_grid(mA,vA,wA,mB,vB,wB,beta,h,won)
    e=max(np.abs(np.r_[mp_[0][0]-bf[0][0],mp_[1][0]-bf[1][0]]).max(), np.abs(np.sqrt(np.r_[mp_[0][1],mp_[1][1]])-np.sqrt(np.r_[bf[0][1],bf[1][1]])).max())
    print(f"{name:26s} max |MP - grid| = {e:.1e}")

## 5. Live run: UTEP vs. North Carolina, Dec 4, 2025
First the season engine (5a) and the lineup recorder (5b) are defined: they produce every player's rating and weight before the match. Then 5c runs the match (about 15 seconds).

### 5a. Season engine: TrueSkill walked forward through every match

In [ ]:
"""Core engine for Nugget 2: player-level TrueSkill with partial-play weights, run walk-forward.
Written in notebook-ready style (heavily commented); the notebook builder pastes these functions in.
"""
import numpy as np
import pandas as pd
from math import sqrt
from scipy.special import log_ndtr, ndtr

LOG_SQRT_2PI = 0.9189385332046727   # log(sqrt(2*pi))


# ------------------------------------------------------------------ truncated-Gaussian helpers
def v_w(t):
    """Truncated-Gaussian correction terms for 'the performance difference d turned out > 0'.
    v(t) = phi(t)/Phi(t)        (how far the mean of d moves, in units of c)
    w(t) = v(t) * (v(t) + t)    (how much the variance of d shrinks, as a fraction of c^2)
    Computed in log-space so that very lopsided matches (t very negative) do not underflow."""
    v = float(np.exp(-0.5 * t * t - LOG_SQRT_2PI - log_ndtr(t)))
    w = v * (v + t)
    return v, min(max(w, 0.0), 1.0 - 1e-12)


def match_update(mu_d, c2, idx_w, w_w, idx_l, w_l, mu, var):
    """Update beliefs (in place) after a match, given the winner/loser rosters.
       mu_d : prior mean of (winner performance - loser performance)   (home advantage already included)
       c2   : prior variance of that difference
       idx_*: array positions of the players on each side;  w_* : their partial-play weights."""
    c = sqrt(c2)
    t = mu_d / c
    v, w = v_w(t)
    for idx, wt, sign in ((idx_w, w_w, +1.0), (idx_l, w_l, -1.0)):
        if idx.size:
            vi = var[idx]
            mu[idx] = mu[idx] + sign * (wt * vi / c) * v           # mean moves up (winner) / down (loser)
            var[idx] = vi * (1.0 - (wt * wt * vi / c2) * w)         # variance always shrinks
    return t


# ------------------------------------------------------------------ data preparation
def prepare_data(pm, matches):
    """Turn the cleaned box scores (pm) and the match table (matches) into integer-indexed arrays."""
    agg = pm.groupby(["Team", "Date", "Player"], as_index=False, dropna=False)["S"].sum()   # sets per player per day
    team_names = sorted(set(pm["Team"]) | set(matches["A"]) | set(matches["B"]))
    team_id = {t: i for i, t in enumerate(team_names)}
    agg["tid"] = agg["Team"].map(team_id).astype(int)
    agg["pid"], player_keys = pd.factorize(agg["tid"].astype(str) + "||" + agg["Player"].astype(str))
    date_list = sorted(set(agg["Date"]) | set(matches["Date"]))
    date_idx = {d: i for i, d in enumerate(date_list)}
    agg["di"] = agg["Date"].map(date_idx).astype(int)

    box = {}                                       # box[day][team] = (player ids, sets played)
    for (di, tid), g in agg.groupby(["di", "tid"]):
        box.setdefault(di, {})[tid] = (g["pid"].values.astype(int), g["S"].values.astype(float))

    season_of = pm.groupby("Date")["SeasonYear"].first()
    season_of_day = {date_idx[d]: int(s) for d, s in season_of.items() if d in date_idx}

    m_di = matches["Date"].map(date_idx).astype(int).values
    m_by_day = {}
    for i, di in enumerate(m_di):
        m_by_day.setdefault(di, []).append(i)

    return dict(
        n_players=len(player_keys), n_teams=len(team_names), team_names=team_names, team_id=team_id,
        player_keys=player_keys, date_list=date_list, box=box, season_of_day=season_of_day,
        days=sorted(set(box) | set(m_by_day)), m_by_day=m_by_day, n_matches=len(matches),
        A=matches["A"].map(team_id).astype(int).values, B=matches["B"].map(team_id).astype(int).values,
        A_won=matches["A_won"].values.astype(int), Home=matches["Home"].values.astype(float),
        m_season=matches["SeasonYear"].values.astype(int), m_di=m_di)


DEFAULT_PARAMS = dict(mu0=25.0, sigma0=25.0 / 3, beta=25.0 / 6, tau_season=25.0 / 300, home=0.0, decay=0.6, rho=1.0)

W_BAR = 6.0   # a volleyball team always has six players on the court


def on_court_weights(share):
    """Turn raw playing-time shares into partial-play weights that always add up to 6 (the six on-court spots).
    Why: a 'share of the team's sets' counts every player who touched a set, so deeper benches add up to more than 6.
    Without this step a team with a longer rotation would get a free head start (sum of weights x prior mean 25)."""
    return W_BAR * share / share.sum()


# ------------------------------------------------------------------ the walk-forward engine
def run_walk_forward(D, P, level="player", update_weights="expected", new_prior="team", lineup="active",
                     end_season=None, record=False, fallback=(6.0, 1.9), return_lineups=False):
    """Walk through every day in order.  For each day:
         (0) register players seen for the first time,
         (1) predict that day's matches from beliefs built ONLY from earlier days,
         (2) update beliefs with that day's results,
         (3) add that day's box scores to the playing-time memory.
       level          : 'player' (team = weighted sum of its players)  or  'team' (each team is one 'player')
       update_weights : 'expected' (same past-playing-time weights as the prediction)
                        'actual'   (today's real participation, when the box score exists)
       new_prior      : 'team'   (a new player starts at her team's current weighted mean rating; 25 if none)  [final model]
                        'global' (a new player starts at 25, as in TrueSkill)
       lineup         : 'active' (only players who have played this season; before the team's first box score
                                  of a season, the players who played last season)                          [final model]
                        'all'    (everyone who has ever played for the team; graduates never leave)
       Everything a prediction uses (ratings, sets, who has played this season) is updated only AFTER that
       day's matches, so each prediction uses information from earlier days only.
       fallback       : (typical sum of weights, typical sum of squared weights) for a team with no history yet"""
    mu0, sig0, beta = P["mu0"], P["sigma0"], P["beta"]
    tau, h, decay, rho = P["tau_season"], P["home"], P["decay"], P["rho"]
    tau_m2 = P.get("tau_match", 0.0) ** 2                                       # drift added per match played
    beta2 = beta * beta
    nU = D["n_players"] if level == "player" else D["n_teams"]
    mu = np.full(nU, float(mu0)); var = np.full(nU, float(sig0) ** 2); seen = np.zeros(nU, bool)
    sets = np.zeros(nU); team_sets = np.zeros(D["n_teams"])
    roster = [[] for _ in range(D["n_teams"])]; roster_np = [None] * D["n_teams"]
    last_season = np.full(nU, -1)                          # last season in which a player had a box score (before today)
    team_played = np.zeros(D["n_teams"], bool)             # has the team had a box score this season (before today)?
    lineups = [None] * D["n_matches"] if return_lineups else None
    fb_mu = fallback[0] * mu0; fb_V = fallback[1] * (sig0 ** 2 + beta2)      # an all-prior team

    n = D["n_matches"]
    p_out = np.full(n, np.nan); muA_out = np.full(n, np.nan); muB_out = np.full(n, np.nan); c_out = np.full(n, np.nan)
    hist = dict(di=[], pid=[], mu=[], var=[]) if record else None
    one = {}                                                                   # team-level: cached 1-element arrays

    def expected_lineup(tid):
        """(player positions, weights) = each known player's share of the team's past sets."""
        if level == "team":
            if tid not in one: one[tid] = (np.array([tid]), np.array([1.0]))
            return one[tid]
        r = roster_np[tid]
        if r is None:
            r = np.array(roster[tid], dtype=int); roster_np[tid] = r
        ts = team_sets[tid]
        if r.size == 0 or ts <= 0:
            return r[:0], np.zeros(0)
        w = np.minimum(sets[r] / ts, 1.0)
        keep = w > 0
        if lineup == "active":                             # drop players who have left (graduated, transferred, ...)
            act = last_season[r] >= cur_season - (0 if team_played[tid] else 1)
            if (keep & act).any(): keep = keep & act
        if not keep.any():
            return r[:0], np.zeros(0)
        return r[keep], on_court_weights(w[keep])

    def stats(idx, w):
        if idx.size == 0:
            return fb_mu, fb_V
        return float(w @ mu[idx]), float((w * w) @ (var[idx] + beta2))

    cur_season = None
    for di in D["days"]:
        season = D["season_of_day"].get(di, cur_season)
        if end_season is not None and season is not None and season > end_season:
            break
        if cur_season is not None and season != cur_season:                  # ---- new season: memory fades, uncertainty grows
            sets *= decay; team_sets *= decay; team_played[:] = False
            mu[seen] = mu0 + rho * (mu[seen] - mu0)
            var[seen] += tau ** 2
        cur_season = season
        todays_box = D["box"].get(di, {})
        todays_matches = D["m_by_day"].get(di, [])

        # (0) first appearances
        if level == "player":
            for tid, (ids, s) in todays_box.items():
                new = ~seen[ids]
                if new.any():
                    nid = ids[new]; seen[nid] = True; var[nid] = sig0 ** 2
                    start = mu0                                              # prior mean of a brand-new player
                    if new_prior == "team":                                  # final model: her team's current weighted
                        li, lw = expected_lineup(tid)                        # mean, from ratings before today
                        if li.size: start = float(lw @ mu[li] / lw.sum())
                    mu[nid] = start
                    roster[tid].extend(nid.tolist()); roster_np[tid] = None
        else:
            for i in todays_matches:
                for tid in (D["A"][i], D["B"][i]):
                    if not seen[tid]: seen[tid] = True; mu[tid] = mu0; var[tid] = sig0 ** 2

        # (1) predict + (2) update, for every match played today
        for i in todays_matches:
            a, b = D["A"][i], D["B"][i]
            ia, wa = expected_lineup(a); ib, wb = expected_lineup(b)
            if return_lineups: lineups[i] = ((ia.copy(), wa.copy()), (ib.copy(), wb.copy()))
            if tau_m2 > 0:                                                       # skill may drift a little between matches
                var[ia] += tau_m2; var[ib] += tau_m2
            mA, VA = stats(ia, wa); mB, VB = stats(ib, wb)
            c2 = VA + VB
            mu_d = mA - mB + h * D["Home"][i]                                   # mean of (perf A - perf B)
            p_out[i] = ndtr(mu_d / sqrt(c2)); muA_out[i] = mA; muB_out[i] = mB; c_out[i] = sqrt(c2)

            if update_weights == "actual" and level == "player":              # use who REALLY played, if we know
                for tid, which in ((a, "a"), (b, "b")):
                    if tid in todays_box:
                        ids, s = todays_box[tid]
                        if which == "a": ia, wa = ids, on_court_weights(s / s.max())
                        else:            ib, wb = ids, on_court_weights(s / s.max())
                mA, VA = stats(ia, wa); mB, VB = stats(ib, wb)
                c2 = VA + VB; mu_d = mA - mB + h * D["Home"][i]
            if D["A_won"][i] == 1:
                match_update(mu_d, c2, ia, wa, ib, wb, mu, var)
            else:
                match_update(-mu_d, c2, ib, wb, ia, wa, mu, var)
            if record:
                for idx in (ia, ib):
                    if idx.size:
                        hist["di"].append(np.full(idx.size, di)); hist["pid"].append(idx)
                        hist["mu"].append(mu[idx].copy()); hist["var"].append(var[idx].copy())

        # (3) memory update with today's box scores
        if level == "player":
            for tid, (ids, s) in todays_box.items():
                sets[ids] += s
                team_sets[tid] += s.max()
                last_season[ids] = cur_season; team_played[tid] = True

    out = dict(p=p_out, muA=muA_out, muB=muB_out, c=c_out, mu=mu, var=var, sets=sets, team_sets=team_sets,
               roster=roster, seen=seen, last_season=last_season, team_played=team_played, lineups=lineups)
    if record:
        out["hist"] = {k: np.concatenate(v) if v else np.array([]) for k, v in hist.items()}
    return out


# ------------------------------------------------------------------ metrics (same definitions as Nugget 3)
def log_loss(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))

def brier(p, y):
    return float(np.mean((p - y) ** 2))

def accuracy(p, y):
    return float(np.mean((p > 0.5) == (y == 1)))


### 5b. Lineup recorder: each player's playing-time weight before each match (the engine's own lineups)


In [ ]:
from math import sqrt

def record_lineups(D, P):
    """For every match, the players of each side and their weights, exactly as the season engine (5a) used them
    for its prediction: past sets only, active lineup (players who have played this season)."""
    return run_walk_forward(D, P, return_lineups=True)["lineups"]


# ------------------------------------------------------------------ 2. build the graph of player-time nodes

### 5c. The match

In [ ]:
import pickle, pandas as pd
D=pickle.load(open('work/D.pkl','rb')); P=pickle.load(open('work/P_tuned.pkl','rb'))
names=D['team_names']; tid=D['team_id']
day=pd.Timestamp('2025-12-04')
cand=[i for i in range(D['n_matches']) if D['date_list'][D['m_di'][i]]==day and {names[D['A'][i]],names[D['B'][i]]}=={'UTEP','North Carolina'}]
i=cand[0]; di=D['m_di'][i]
print('match',i,'A=',names[D['A'][i]],'B=',names[D['B'][i]],'A_won',D['A_won'][i],'Home',D['Home'][i])
D2=dict(D); D2['days']=[d for d in D['days'] if d<di]
o=run_walk_forward(D2,P)
lu=record_lineups(D,P)[i]
(ia,wa),(ib,wb)=lu
mu,var=o['mu'],o['var']
keys=D['player_keys']
def show(ids,w,label):
    print(label)
    for p,ww in sorted(zip(ids,w),key=lambda x:-x[1]): print(f"   {keys[p].split('||')[1]:22s} w={ww:.2f} mu={mu[p]:6.2f} sd={np.sqrt(var[p]):.2f}")
show(ia,wa,names[D['A'][i]]); show(ib,wb,names[D['B'][i]])
beta=P["beta"]; h=0.0   # neutral site (played in Madison); the data wrongly marks it as a UTEP home game
won=bool(D['A_won'][i])
mA=wa@mu[ia]; mB=wb@mu[ib]; c=np.sqrt((wa**2)@(var[ia]+beta**2)+(wb**2)@(var[ib]+beta**2))
from scipy.special import ndtr
print(f"team means {mA:.1f} vs {mB:.1f}, c={c:.2f}, P(A wins)={ndtr((mA-mB+h)/c):.3f}, h={h}")
log=[]
mpA,mpB=message_passing(mu[ia],var[ia],wa,mu[ib],var[ib],wb,beta,h,won,log)
cfA,cfB=closed_form(mu[ia],var[ia],wa,mu[ib],var[ib],wb,beta,h,won)
mcA,mcB=monte_carlo(mu[ia],var[ia],wa,mu[ib],var[ib],wb,beta,h,won,N=3_000_000)
d=lambda a,b: max(np.abs(a[0]-b[0]).max(),np.abs(np.sqrt(a[1])-np.sqrt(b[1])).max())
print('max|MP-CF|',d(mpA,cfA),d(mpB,cfB)); print('max|MP-MC| (mu or sd)',d(mpA,mcA),d(mpB,mcB))
for nm,m,v in log[:3]+[l for l in log if l[0].startswith(('sum->t','diff->d','d marginal','trunc'))]: print(f"  {nm:35s} mean={m:8.3f} sd={np.sqrt(v):.3f}")

### 5d. Show how the weights are calculated (North Carolina, before Dec 4)
Step 1: each player's share of the team's sets in earlier matches (old seasons multiplied by decay). Step 2: keep only players who have played this season. Step 3: rescale so the team's weights sum to 6. This is exactly the lineup the engine (5a) uses for the prediction.


In [ ]:
# 5d. How the weights are calculated: North Carolina, before the Dec 4, 2025 match
import pickle, numpy as np, pandas as pd
D = pickle.load(open('work/D.pkl', 'rb')); P = pickle.load(open('work/P_tuned.pkl', 'rb'))
team = D['team_id']['North Carolina']
cut = pd.Timestamp('2025-12-04')                      # only matches BEFORE this day are used
D2 = dict(D); D2['days'] = [di for di in D['days'] if D['date_list'][di] < cut]
o = run_walk_forward(D2, P)                           # the engine's playing-time memory the night before

ids = np.array(o['roster'][team], dtype=int)          # everyone who has ever played for the team
share = np.minimum(o['sets'][ids] / o['team_sets'][team], 1.0)          # step 1: share of the team's sets
active = o['last_season'][ids] >= 2025                                   # step 2: has played this season
keep = (share > 0) & active
weight = 6 * share[keep] / share[keep].sum()                             # step 3: rescale so the team sums to 6

names = [D['player_keys'][p].split('||')[1] for p in ids[keep]]
df = pd.DataFrame({'player': names, 'sets (decayed)': o['sets'][ids[keep]], 'raw share': share[keep], 'weight': weight})
print(df.sort_values('weight', ascending=False).head(12).round(3).to_string(index=False))
print(f"\nplayers ever on the roster: {len(ids)}   played this season: {keep.sum()}   team sets (decayed): {o['team_sets'][team]:.1f}")
print(f"sum of raw shares: {share[keep].sum():.2f}   ->   rescaled by 6 / {share[keep].sum():.2f}   ->   sum of weights: {weight.sum():.2f}")


## 6. Results: TrueSkill vs. the RPI on the 2025 season
Three columns on the results slides:
- **RPI**: recomputed every day from this season's results (6c).
- **TrueSkill, same information**: the same algorithm with one rating per team (the 1 vs. 1 case), this season's wins and losses only, no home court (6c, 6d). It sees exactly what the RPI sees.
- **TrueSkill, full**: the 6 vs. 6 player-level model from Nugget 3 (final model: active lineup, new players start at their team's average; `work/P_tuned.pkl`, tuned on 2024 in section 7). It is recomputed below in a few seconds.

6a and 6b take seconds. 6c and 6d rebuild the RPI and same-information predictions from scratch (several minutes each); you do not need to run them. Needs 5a to have been run.

### 6a. Accuracy, log-loss, bootstrap intervals

In [ ]:
import pickle,numpy as np,pandas as pd
from scipy.special import expit
from scipy.optimize import minimize_scalar
D=pickle.load(open('work/D.pkl','rb')); R=pickle.load(open('cmp/rpi_wf.pkl','rb')); T=pickle.load(open('cmp/ts_wf.pkl','rb'))
y=D['A_won'].astype(float); S=D['m_season']; dates=np.array(D['date_list'])[D['m_di']]
dR=R['rpi_A']-R['rpi_B']; k=np.minimum(R['nA'],R['nB'])
def li(p): p=np.clip(p,1e-12,1-1e-12); return -(y*np.log(p)+(1-y)*np.log(1-p))
fit=(S==2024)&(k>=5)
b=minimize_scalar(lambda b: li(expit(b*dR))[fit].mean(),bounds=(0,200),method='bounded').x
# TrueSkill full = the player-level model from Nugget 3, walked forward through every match (a few seconds)
P_player=pickle.load(open('work/P_tuned.pkl','rb'))
p_player=run_walk_forward(D,P_player)['p']
M={'RPI':expit(b*dR),'RPI+home':T['rpi_home'],'TS same info':T['fair_tuned'],'TS full':p_player}
rng=np.random.default_rng(1)
def ci(d):
    bs=[d[rng.integers(0,len(d),len(d))].mean() for _ in range(2000)]; return np.percentile(bs,[2.5,97.5])
for name,mask in [('2025 both>=5',(S==2025)&(k>=5)),('2025 before Oct15',(S==2025)&(k>=5)&(dates<pd.Timestamp('2025-10-15'))),('2025 Oct15+',(S==2025)&(k>=5)&(dates>=pd.Timestamp('2025-10-15'))),('Tourney',(S==2025)&(dates>=pd.Timestamp('2025-12-04')))]:
    m=mask&~np.isnan(dR)
    print('==',name,'n=',m.sum())
    for nm,p in M.items():
        print(f'  {nm:14s} LL={li(p)[m].mean():.4f} acc={np.mean((p[m]>.5)==(y[m]==1)):.4f}')
    for a,c in [('TS same info','RPI'),('TS full','RPI'),('TS full','RPI+home')]:
        d=li(M[a])[m]-li(M[c])[m]; acc=((M[a][m]>.5)==(y[m]==1)).astype(float)-((M[c][m]>.5)==(y[m]==1))
        dis=m&((M[a]>.5)!=(M[c]>.5))
        print(f'  {a} - {c}: dLL={d.mean():+.4f} CI={np.round(ci(d),4)} dAcc={acc.mean():+.4f} CI={np.round(ci(acc),4)}; disagree n={dis.sum()} {a} right {np.mean((M[a][dis]>.5)==(y[dis]==1)):.3f}')

### 6b. Rankings at the end of the regular season, and UTEP vs. North Carolina (a few seconds)

In [ ]:
# Rankings at the end of the regular season, and the UTEP vs. North Carolina predictions (TrueSkill full = player-level)
import pickle, numpy as np, pandas as pd
from scipy.special import expit
from scipy.stats import spearmanr, norm
from scipy.optimize import minimize_scalar
D = pickle.load(open('work/D.pkl', 'rb')); P = pickle.load(open('work/P_team.pkl', 'rb'))
T = pickle.load(open('cmp/ts_wf.pkl', 'rb')); R = pickle.load(open('cmp/rpi_wf.pkl', 'rb'))
A, B, y, S, DI = D['A'], D['B'], D['A_won'], D['m_season'], D['m_di']
dates = D['date_list']; names = D['team_names']; nT = D['n_teams']

def rpi_from(games):
    """RPI = 0.25 WP + 0.50 OWP + 0.25 OOWP from a list of (team a, team b, a won?) games."""
    W = np.zeros(nT); N = np.zeros(nT); pw = {}; pn = {}
    for a, b, aw in games:                                  # WP: wins and games for every team
        W[a] += aw; W[b] += 1 - aw; N[a] += 1; N[b] += 1
        pw[(a, b)] = pw.get((a, b), 0) + aw; pw[(b, a)] = pw.get((b, a), 0) + 1 - aw
        pn[(a, b)] = pn.get((a, b), 0) + 1;  pn[(b, a)] = pn.get((b, a), 0) + 1
    with np.errstate(invalid='ignore', divide='ignore'): wp = W / N
    os_ = np.zeros(nT); oc = np.zeros(nT)                    # OWP: opponents' win %, leaving out games vs. this team
    for a, b, aw in games:
        for t, o in ((a, b), (b, a)):
            n = N[o] - pn[(o, t)]
            if n > 0: os_[t] += (W[o] - pw[(o, t)]) / n; oc[t] += 1
    with np.errstate(invalid='ignore', divide='ignore'): owp = os_ / oc
    s2 = np.zeros(nT); c2 = np.zeros(nT)                     # OOWP: opponents' OWP
    for a, b, aw in games:
        for t, o in ((a, b), (b, a)):
            if not np.isnan(owp[o]): s2[t] += owp[o]; c2[t] += 1
    with np.errstate(invalid='ignore', divide='ignore'): oowp = s2 / c2
    return 0.25*wp + 0.5*owp + 0.25*oowp, N, (wp, owp, oowp)

cut = pd.Timestamp('2025-11-30')                              # end of the regular season
idx = [i for i in range(len(A)) if S[i] == 2025 and dates[DI[i]] < cut]
rpi, N, (wp, owp, oowp) = rpi_from([(A[i], B[i], int(y[i])) for i in idx])

# TrueSkill, full = the player-level model. A team's strength is its expected lineup, sum of w_i * mu_i (Algorithm 1, line 1)
P_player = pickle.load(open('work/P_tuned.pkl', 'rb'))
D2 = dict(D); D2['days'] = [di for di in D['days'] if dates[di] < cut]
oF = run_walk_forward(D2, P_player)
def lineup(o, t, season=2025):
    r = np.array(o['roster'][t], dtype=int); ts = o['team_sets'][t]
    if r.size == 0 or ts <= 0: return r[:0], np.zeros(0)
    w = np.minimum(o['sets'][r] / ts, 1.0); keep = w > 0
    act = o['last_season'][r] >= season - (0 if o['team_played'][t] else 1)   # active lineup, as in the engine (5a)
    if (keep & act).any(): keep = keep & act
    if not keep.any(): return r[:0], np.zeros(0)
    return r[keep], on_court_weights(w[keep])
def strength(o, t):
    r, w = lineup(o, t); return float(w @ o['mu'][r]) if r.size else np.nan
muF = np.array([strength(oF, t) for t in range(nT)])
D3 = dict(D); D3['days'] = [di for di in D['days'] if dates[di] < cut and D['season_of_day'][di] == 2025]
muS = run_walk_forward(D3, T['P_fair'], level='team')['mu']                      # TrueSkill, this season only

played = N > 0
df = pd.DataFrame({'team': np.array(names)[played], 'W-L': [f"{int(round(w*n))}-{int(n-round(w*n))}" for w, n in zip(wp[played], N[played])],
                   'WP': wp[played], 'OWP': owp[played], 'OOWP': oowp[played], 'RPI': rpi[played],
                   'RPI rank': pd.Series(rpi[played]).rank(ascending=False).astype(int).values,
                   'TS same info': pd.Series(muS[played]).rank(ascending=False).astype(int).values,
                   'TS full': pd.Series(muF[played]).rank(ascending=False).astype(int).values})
print(df[df.team.isin(['UTEP', 'North Carolina'])].set_index('team').round(3).to_string())
# Largest disagreements: among teams with at least 20 D1 matches and in either top 64, three largest gaps each way
cand = df[(N[played] >= 20) & ((df['RPI rank'] <= 64) | (df['TS full'] <= 64))].copy()
cand['gap'] = cand['TS full'] - cand['RPI rank']
print('\nRPI ranks higher than TrueSkill full:'); print(cand.nlargest(3, 'gap')[['team', 'W-L', 'RPI rank', 'TS same info', 'TS full']].to_string(index=False))
print('TrueSkill full ranks higher than the RPI:'); print(cand.nsmallest(3, 'gap')[['team', 'W-L', 'RPI rank', 'TS same info', 'TS full']].to_string(index=False))
print(f"\nRank correlation with the RPI: TrueSkill full {spearmanr(df.RPI, muF[played], nan_policy='omit').correlation:.2f}, same info {spearmanr(df.RPI, muS[played]).correlation:.2f}")

# Pre-match P(North Carolina wins), Dec 4, 2025, neutral site
i = [j for j in range(len(A)) if S[j] == 2025 and {names[A[j]], names[B[j]]} == {'UTEP', 'North Carolina'}][0]
unc_is_A = names[A[i]] == 'North Carolina'
dR = R['rpi_A'] - R['rpi_B']; k = np.minimum(R['nA'], R['nB']); fit = (S == 2024) & (k >= 5) & ~np.isnan(dR)
yy = y.astype(float)
li = lambda p: -(yy*np.log(np.clip(p, 1e-12, 1)) + (1-yy)*np.log(np.clip(1-p, 1e-12, 1)))
slope = minimize_scalar(lambda b: li(expit(b*dR))[fit].mean(), bounds=(0, 200), method='bounded').x
p_rpi = expit(slope*dR[i]); p_same = T['fair_tuned'][i]
D4 = dict(D); D4['days'] = [di for di in D['days'] if dates[di] < pd.Timestamp('2025-12-04')]
o = run_walk_forward(D4, P_player); a, b = D['team_id']['North Carolina'], D['team_id']['UTEP']
(ra, wa), (rb, wb) = lineup(o, a), lineup(o, b); b2 = P_player['beta']**2
p_full_unc = norm.cdf((wa @ o['mu'][ra] - wb @ o['mu'][rb]) /
                      np.sqrt((wa**2) @ (o['var'][ra] + b2) + (wb**2) @ (o['var'][rb] + b2)))   # h = 0, neutral site (same as 5c)
f = (lambda p: p) if unc_is_A else (lambda p: 1 - p)
print(f"\nP(North Carolina wins):  RPI {f(p_rpi):.0%}   TrueSkill same info {f(p_same):.0%}   TrueSkill full {p_full_unc:.0%}")

### 6c. (optional, slow) Rebuild: walk-forward RPI and TrueSkill predictions

In [ ]:
import sys,pickle,numpy as np,pandas as pd
D=pickle.load(open('work/D.pkl','rb')); P=pickle.load(open('work/P_team.pkl','rb'))
A,B,y,S,DI,H=D['A'],D['B'],D['A_won'].astype(bool),D['m_season'],D['m_di'],D['Home']
dates=D['date_list']; nT=D['n_teams']
print('Home values',np.unique(H,return_counts=True))
# ---------- walk-forward RPI (season to date, D1 vs D1, before the match day) ----------
rpi_A=np.full(len(A),np.nan); rpi_B=np.full(len(A),np.nan); nA=np.zeros(len(A)); nB=np.zeros(len(A))
def rpi_from(games):
    W=np.zeros(nT); N=np.zeros(nT); pw={}; pn={}
    for a,b,aw in games:
        W[a]+=aw; W[b]+=1-aw; N[a]+=1; N[b]+=1
        pw[(a,b)]=pw.get((a,b),0)+aw; pw[(b,a)]=pw.get((b,a),0)+1-aw
        pn[(a,b)]=pn.get((a,b),0)+1; pn[(b,a)]=pn.get((b,a),0)+1
    with np.errstate(invalid='ignore',divide='ignore'): wp=W/N
    os=np.zeros(nT); oc=np.zeros(nT)
    for a,b,aw in games:
        for t,o in ((a,b),(b,a)):
            n=N[o]-pn[(o,t)]
            if n>0: os[t]+=(W[o]-pw[(o,t)])/n; oc[t]+=1
    with np.errstate(invalid='ignore',divide='ignore'): owp=os/oc
    s2=np.zeros(nT); c2=np.zeros(nT)
    for a,b,aw in games:
        for t,o in ((a,b),(b,a)):
            if not np.isnan(owp[o]): s2[t]+=owp[o]; c2[t]+=1
    with np.errstate(invalid='ignore',divide='ignore'): oowp=s2/c2
    return 0.25*wp+0.5*owp+0.25*oowp, N, (wp,owp,oowp)
for season in (2024,2025):
    days=sorted(set(DI[S==season]))
    games=[]
    for di in days:
        idx=np.where(DI==di)[0]
        if games:
            r,N,_=rpi_from(games)
            rpi_A[idx]=r[A[idx]]; rpi_B[idx]=r[B[idx]]; nA[idx]=N[A[idx]]; nB[idx]=N[B[idx]]
        games+= [(A[i],B[i],int(y[i])) for i in idx]
    if season==2025:
        final_rpi,_,comps=rpi_from([g for g in games if dates[DI[0]]<=dates[DI[0]]])
pickle.dump(dict(rpi_A=rpi_A,rpi_B=rpi_B,nA=nA,nB=nB),open('cmp/rpi_wf.pkl','wb'))
# ---------- TrueSkill team-level: full and "fair" (current season only, no home) ----------
full=run_walk_forward(D,P,level='team')['p']
Pf=dict(P,home=0.0)
fair=np.full(len(A),np.nan)
for season in (2024,2025):
    D2=dict(D); D2['days']=[di for di in D['days'] if D['season_of_day'][di]==season]
    fair_s=run_walk_forward(D2,Pf,level='team')['p']; m=S==season; fair[m]=fair_s[m]
pickle.dump(dict(full=full,fair=fair),open('cmp/ts_wf.pkl','wb'))
print('done')

### 6d. (optional, slow) Tune same-information TrueSkill; add a home-court term to the RPI

In [ ]:
import sys,pickle,numpy as np
from scipy.special import expit
from scipy.optimize import minimize
D=pickle.load(open('work/D.pkl','rb')); P=pickle.load(open('work/P_team.pkl','rb')); R=pickle.load(open('cmp/rpi_wf.pkl','rb'))
y=D['A_won'].astype(float); S=D['m_season']; k=np.minimum(R['nA'],R['nB']); H=D['Home']
def ll(p,yy): p=np.clip(p,1e-12,1-1e-12); return -np.mean(yy*np.log(p)+(1-yy)*np.log(1-p))
def fair(Pf,season):
    D2=dict(D); D2['days']=[di for di in D['days'] if D['season_of_day'][di]==season]
    return run_walk_forward(D2,Pf,level='team')['p']
m24=(S==2024)&(k>=5)
res=[]
for s0 in [2,3,4,5,6,8.33,12]:
  for tm in [0,0.5,1]:
    Pf=dict(P,home=0.0,sigma0=s0,tau_match=tm); p=fair(Pf,2024); res.append((ll(p[m24],y[m24]),s0,tm))
res.sort(); print(res[:5])
best=res[0]; Pf=dict(P,home=0.0,sigma0=best[1],tau_match=best[2])
p25=fair(Pf,2025); p24=fair(Pf,2024)
# RPI + home logistic fitted on 2024
dR=R['rpi_A']-R['rpi_B']
f=lambda w: ll(expit(w[0]*dR[m24]+w[1]*H[m24]),y[m24])
w=minimize(f,[16,0.2]).x; print('RPI+home weights',w)
pRH=expit(w[0]*dR+w[1]*H)
T=pickle.load(open('cmp/ts_wf.pkl','rb')); T['fair_tuned']=np.where(S==2025,p25,p24); T['rpi_home']=pRH; T['P_fair']=Pf
pickle.dump(T,open('cmp/ts_wf.pkl','wb'))

## 7. Learning the settings by maximum likelihood: Algorithm 3 (CoordinateSearch) — about 1 minute
Objective (Nugget 1): maximize $L(\theta)=\sum_{\text{2024 matches}}\log P(\text{result}\mid\text{earlier results},\theta)$; mean log loss $=-L/n$, so the lowest log loss is the maximum likelihood.
Method (Algorithm 3): coordinate search, one setting at a time over its fixed grid, moving only on a strict improvement, repeated until a full round changes nothing. Each evaluation of $L$ is one SeasonScore run (Algorithm 2 = `run_walk_forward` from 5a): ratings run 2022–2024, only 2024 matches are scored, each predicted before it is used; 2025 is never touched. $\beta$ is fixed because it only sets the scale.
This is an approximate E-step plus coordinate search, not EM, so there is no EM ascent guarantee; the search itself never lowers the $L$ it computes. The likelihood is fairly flat near the optimum: a finer grid and a gradient method (7b) find slightly higher validation likelihoods with different settings. Needs 5a to have been run.


In [ ]:
# 7. Algorithm 3: learn the settings by maximum likelihood (loop to convergence), about 1-2 minutes
import pickle, time, numpy as np
D = pickle.load(open('work/D.pkl', 'rb')); y = D['A_won']
val = D['m_season'] == 2024          # matches used to CHOOSE the settings
tst = D['m_season'] == 2025          # matches used only ONCE, at the end, to test

runs = []                            # every SeasonScore execution (one full walk through 2022-2024)
def neg_ll(P):                       # mean negative log-likelihood (log-loss) of the 2024 predictions
    runs.append(tuple(P[k] for k in grids))
    out = run_walk_forward(D, P, end_season=2024)
    return log_loss(out["p"][val], y[val])

grids = dict(sigma0=[2, 3, 4, 5, 6, 8.33, 10, 12],
             tau_season=[0, 0.25, 0.5, 1, 1.5, 2, 3],
             home=[0, 0.5, 1, 1.5, 2, 3, 4],
             decay=[0.2, 0.4, 0.6, 0.8, 1.0],
             rho=[0.6, 0.8, 0.9, 1.0])
P = dict(DEFAULT_PARAMS); t0 = time.time()
cur = neg_ll(P); print(f"start (TrueSkill defaults): log-loss {cur:.5f}")
it = 0
while True:                          # loop to convergence
    it += 1; changed = []
    for name, vals in grids.items():
        res = []
        for v in vals:
            Q = dict(P); Q[name] = v; res.append((neg_ll(Q), v))
        ll, best = min(res)
        if ll < cur:                 # strict improvement only (Algorithm 3, line 6)
            if best != P[name]: changed.append(f"{name} {P[name]:g}->{best:g}")
            P[name] = best; cur = ll
    print(f"round {it}: log-loss {cur:.5f}  " + "  ".join(f"{k}={P[k]:g}" for k in grids) +
          ("   changed: " + ", ".join(changed) if changed else "   (no change: converged)"))
    if not changed: break
print(f"converged after {it} rounds: {len(runs)} SeasonScore runs ({len(set(runs))} distinct settings), {time.time()-t0:.0f} s")
out = run_walk_forward(D, P)
for nm, m in (("validation 2024", val), ("test 2025", tst)):
    p = out["p"][m]; print(f"{nm}: log-loss {log_loss(p, y[m]):.4f}  accuracy {accuracy(p, y[m]):.1%}  (all {m.sum()} matches)")

# the settings used everywhere else (Sections 5c, 6) come from this procedure
old = pickle.load(open('work/P_tuned.pkl', 'rb'))
print("same as work/P_tuned.pkl:", all(abs(old[k] - P[k]) < 1e-12 for k in grids))


### 7b. (optional) How flat is the likelihood near the choice? A gradient method and a finer grid
L-BFGS-B (finite-difference gradients) on the same 2024 log-loss, from the TrueSkill defaults (about 1–2 minutes), then a finer grid of 540 combinations around the choice (several minutes). Both find slightly better 2024 scores (by 0.0002–0.0004) with different settings: the predictive likelihood is fairly flat around several good solutions, so we do not claim the methods recover the same θ.


In [ ]:
# 7b-1. gradient method (L-BFGS-B), about 1-2 minutes
import pickle, time, numpy as np
from scipy.optimize import minimize
D = pickle.load(open('work/D.pkl', 'rb')); y = D['A_won']; val = D['m_season'] == 2024; tst = D['m_season'] == 2025
P_grid = pickle.load(open('work/P_tuned.pkl', 'rb'))
names = ['sigma0', 'tau_season', 'home', 'decay', 'rho']; lo = [1, 0, 0, 0.05, 0.3]; hi = [15, 3, 5, 1, 1]
def P_of(x): P = dict(DEFAULT_PARAMS); P.update(dict(zip(names, x))); return P
n = [0]
def f(x):
    n[0] += 1; return log_loss(run_walk_forward(D, P_of(x), end_season=2024)['p'][val], y[val])
def test(x): o = run_walk_forward(D, P_of(x)); return log_loss(o['p'][tst], y[tst])
t0 = time.time()
x0 = np.array([DEFAULT_PARAMS[k] for k in names])
r = minimize(f, x0, method='L-BFGS-B', bounds=list(zip(lo, hi)), options=dict(eps=1e-3, maxiter=40))
print('L-BFGS-B from defaults:', dict(zip(names, np.round(r.x, 3))), ' 2024 log-loss %.5f  2025 log-loss %.5f' % (r.fun, test(r.x)),
      ' iterations', r.nit, 'evaluations', n[0], round(time.time() - t0), 's')
xg = np.array([P_grid[k] for k in names])
fg = f(xg); print('coordinate-search choice:', dict(zip(names, xg)), ' 2024 log-loss %.5f  2025 log-loss %.5f' % (fg, test(xg)))
g = []
for i in range(5):
    e = np.zeros(5); e[i] = 1e-2
    g.append((fg - f(xg - e)) / 1e-2 if xg[i] + 1e-2 > hi[i] else (f(xg + e) - f(xg - e)) / 2e-2)
print('finite-difference gradient of the 2024 log-loss at the choice:', dict(zip(names, np.round(g, 5))))


In [ ]:
# 7b-2. finer grid of 540 combinations, several minutes
import sys, pickle, itertools, time, numpy as np
from multiprocessing import Pool
D = pickle.load(open('work/D.pkl', 'rb')); P0 = pickle.load(open('work/P_tuned.pkl', 'rb'))
mv = D['m_season'] == 2024; mt = D['m_season'] == 2025; y = D['A_won']
S = [3, 3.5, 4, 4.5, 5, 6]; H = [2, 2.5, 3, 3.5, 4]; DE = [0.6, 0.8, 1.0]; T = [0.5, 1, 1.5]; R = [0.9, 1.0]
combos = list(itertools.product(S, H, DE, T, R))           # 6 x 5 x 3 x 3 x 2 = 540
def f(c):
    s, h, de, t, r = c; P = dict(P0, sigma0=s, home=h, decay=de, tau_season=t, rho=r)
    p = run_walk_forward(D, P)["p"]
    return c, log_loss(p[mv], y[mv]), log_loss(p[mt], y[mt])
if __name__ == "__main__":
    t0 = time.time()
    with Pool() as pool: res = pool.map(f, combos)
    pickle.dump(res, open('lec/grid_res.pkl', 'wb'))
    res.sort(key=lambda r: r[1])
    print(len(res), "combinations (sigma0, home, decay, tau, rho)", round(time.time() - t0), "s")
    for r in res[:5]: print("  ", r[0], "2024 %.5f  2025 %.5f" % (r[1], r[2]))
    pick = (P0['sigma0'], P0['home'], P0['decay'], P0['tau_season'], P0['rho'])
    cs = [r for r in res if r[0] == pick][0]
    print("coordinate-search pick", pick, "2024 %.5f  2025 %.5f" % (cs[1], cs[2]), "rank", res.index(cs) + 1, "of", len(res),
          " 2024 gap to #1: %.5f" % (cs[1] - res[0][1]))
